# Setup

In [1]:
import numpy as np, os, json
import requests
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer

c:\Users\hp!\OneDrive\Desktop\Plaksha\Term 1\Python Programming\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# client= # API key
embedder=SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11696.27it/s]


## Load

Loading papers from data folder

In [3]:
def load_corpus(path='research_papers'):
    docs=[]
    for fname in sorted(os.listdir(path)): 
        # os.listdir(path) returns a list of all files and folders inside the directory.
        # being sorted lexicographically.
        full_path=os.path.join(path,fname)
        # path becomes research_paper/file1.pdf

        if fname.endswith('pdf'):
            reader=PdfReader(full_path)
            # print(f'{reader}: {fname}: Length: {len(reader.pages)} pages')
            f_text=reader.pages[0].extract_text()
            # print(f'First 500 characers: {f_text[:500]}')
            # New Line for headingd. This also all extracts text
            text='\n'.join(page.extract_text() for page in reader.pages)
            # Test to check what text is extracted
            # print(text)
        elif fname.endswith('.txt'):
            with open(full_path,'r',encoding='utf-8') as f:
                text=f.read()
        else:
            continue
        # os.path.splitext(fname) splits the filename into:(
            # "A survey on multi-objective hyperparameter optimization algorithms for machine learning",
            # ".pdf"
        # )

        docs.append({'id':os.path.splitext(fname)[0],'title':fname,'text':text})
    return docs
        

In [4]:
# testing
corpus=load_corpus()

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/JLPIYQ+TimesNewRomanPSMT', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(42, 0, 2227279675120), '/LastChar': 117, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [250, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 500, 500, 500, 500, 0, 500, 500, 500, 500, 500, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 722, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 444, 500, 444, 0, 444, 333, 0, 0, 278, 0, 0, 0, 778, 500, 500, 0, 0, 333, 389, 278, 500]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/TimesNewRomanPSMT2', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(44, 0, 2227279675120), '/LastChar': 117, '/Subtype': '/Type1', '/Type': '/Font', '/

In [5]:
# difference between id and title are that title contains the extension as well as splitting the text
# divides the title into title and the extension
for corp in corpus:
    print(corp['id'])
    print(corp['title'])

A Systematic Review of the Whale Optimization Algorithm - Theoretical Foundation, Improvements, and Hybridizations
A Systematic Review of the Whale Optimization Algorithm - Theoretical Foundation, Improvements, and Hybridizations.pdf
A survey on multi-objective hyperparameter optimization algorithms for machine learning
A survey on multi-objective hyperparameter optimization algorithms for machine learning.pdf
Application of machine learning, deep learning and optimization
Application of machine learning, deep learning and optimization.pdf
On Hyperparameter Optimization of Machine Learning Methods Using a Bayesian Optimization Algorithm to Predict Work Travel Mode Choice
On Hyperparameter Optimization of Machine Learning Methods Using a Bayesian Optimization Algorithm to Predict Work Travel Mode Choice.pdf
Puma optimizer -  a novel metaheuristic optimization algorithm and its application in machine learning
Puma optimizer -  a novel metaheuristic optimization algorithm and its applicat

## Chunking

In [6]:
print(type(corpus))

<class 'list'>


In [7]:
# list of keys in corpus
print(corpus[0].keys())

dict_keys(['id', 'title', 'text'])


In [8]:
def chunk_text(text,chunk_size=300,overlap=30):
    # words: list of words
    words=text.split()
    # print(words,'\n')
    # chunks: stores text chunks
    # start: index of first word of current chunk
    chunks,start=[],0
    while start<len(words):
        end=start+chunk_size
        chunks.append(' '.join(words[start:end]))
        # to stop chunking when we reach end.
        if end>=len(words):
            break
        # determining where next chunk starts. It starts from the end of the current chunk minus the overlap.
        start+=chunk_size-overlap
    return chunks

In [9]:
# # Test
# a=chunk_text(corpus[1]['text'])
# print(a)

In [10]:
chunk_records=[]
for doc in corpus:
    # chunks the current paper and get chunk number and actual chunk text
    for i,c in enumerate(chunk_text(doc['text'])):
        # each chunk becomes a dictionary
        chunk_records.append({'chunk_id':f'{doc['id']}_c{i}',
                              'doc_id':f'{doc['id']}',
                              'doc_title':f'{doc['title']}',
                              'text':c})

# Number of chunks obtained
print(f'{len(chunk_records)} chunks from {len(corpus)} papers')

435 chunks from 7 papers


In [11]:
def embed_texts(texts):
    # convert text to numerical data
    vecs=embedder.encode(texts,convert_to_numpy=True)
    print(vecs,'\n')
    # calculate length of each embedding vector
    norms=np.linalg.norm(vecs,axis=1,keepdims=True)
    print(norms)
    norms[norms==0]=1e-8 # prevent decision by 0; replace 0 with very tiny number
    return vecs/norms # normalize vectors

In [12]:
# store all embeddings in chunk matrix
chunk_matrix=embed_texts([c['text'] for c in chunk_records])
print(chunk_matrix,'\n')
print(chunk_matrix.shape)

[[-5.10611869e-02  1.14187583e-01  4.79632802e-02 ... -1.77145787e-02
  -6.43941835e-02  3.71739529e-02]
 [-8.52738246e-02  5.22756018e-02 -5.75622357e-02 ...  4.34695259e-02
   3.07562575e-02 -1.79001857e-02]
 [-2.77046897e-02  3.36401798e-02  1.76290926e-02 ... -2.62215100e-02
  -5.75903840e-02 -8.45483970e-03]
 ...
 [-7.53695220e-02  8.80982820e-03  3.17649394e-02 ... -7.74805099e-02
  -2.17327643e-02  5.91890588e-02]
 [ 5.86928846e-03 -2.27237605e-02  2.72563793e-05 ... -1.03546135e-01
  -6.32933974e-02  4.76739369e-02]
 [ 5.77335712e-03 -4.18651700e-02  5.86352795e-02 ... -7.04207793e-02
  -6.63428232e-02  4.37705480e-02]] 

[[0.99999994]
 [1.        ]
 [0.99999994]
 [1.        ]
 [1.        ]
 [0.99999994]
 [1.        ]
 [1.        ]
 [1.        ]
 [1.        ]
 [0.9999999 ]
 [1.        ]
 [1.        ]
 [1.        ]
 [0.99999994]
 [1.        ]
 [1.        ]
 [0.99999994]
 [1.        ]
 [0.99999994]
 [1.        ]
 [1.        ]
 [1.        ]
 [1.        ]
 [0.99999994]
 [1.        

In [13]:
def retrieve(query,k=3):
    '''
    Retrieves K most relevant text chunks from paper for a given query. In this case it
    will return 3 most relevant chunks.
    '''
    q_vec=embed_texts([query])[0] # [0] because its a row vector
    sims=chunk_matrix@q_vec # sims is similarity between query and chunks
    print(sims.shape)
    # This statement returns the indices of the top k most similar chunks. 
    top_idx=np.argsort(-sims)[:k] # usually argsort returns indices of sorted array in ascending order. We want descending order so we multiply by -1.
    return [{**chunk_records[i], # copis all fields of the chunk record to another dictionary and adds a new field 'score' which is the similarity score of the chunk with the query..
              'score': float(sims[i])} for i in top_idx] # return the chunk records along with the similarity score


In [14]:
SYSTEM_PROMPT=('Answer using ONLY the provided context below (research papers and, if present, web results).'
"If the context doesn\'t contain the answer, just SAY SO."
'Cite each fact with its source: [Paper: filename] or [Web: title].'
'Also summarize the answer in 2-3 lines.')

In [15]:

from groq import Groq
from dotenv import load_dotenv

load_dotenv()

api_key=os.getenv('GROQ_API_KEY')
model=os.getenv('GROQ_MODEL')

client=Groq(api_key=api_key)

In [ ]:
## Web search fallback (when the corpus lacks the answer)
WEB_SEARCH_THRESHOLD=0.40  # top retrieval score below this -> search the web
def search_web(query,max_results=5):
    '''
    Searches the web via the Tavily API when the retrieved score is too low.
    Returns a list of {title,url,content}; [] if key missing or request failed.
    '''
    tavily_key=os.getenv('TAVILY_API_KEY')
    if not tavily_key:
        print('[web_search] TAVILY_API_KEY not set; skipping web search.')
        return []
    resp=requests.post('https://api.tavily.com/search',
        json={'api_key':tavily_key,'query':query,'max_results':max_results,'search_depth':'basic'},
        timeout=15)
    if resp.status_code!=200:
        print(f'[web_search] Tavily error {resp.status_code}: {resp.text[:300]}')
        return []
    data=resp.json()
    return [{'title':r.get('title',''),'url':r.get('url',''),'content':r.get('content','')}
            for r in data.get('results',[])]


In [16]:
def ask(query,k=3):
    # retrieves top k relevant chunks
    retrieved=retrieve(query,k)
    # cosine similarity of the single best chunk; low => corpus may not cover the question
    top_score=max(r['score'] for r in retrieved)
    web_search_used=top_score<WEB_SEARCH_THRESHOLD
    # formats the chunks
    # [Paper A]
    # RAG combines...
    context='\n\n'.join(f'[{ret['doc_title']}]\n{ret['text']}' for ret in retrieved)
    web_results=[]
    if web_search_used:
        web_results=search_web(query)
        if web_results:
            web_context='\n\n'.join(f"[Web: {r['title']}]\n{r['url']}\n{r['content'][:500]}" for r in web_results)
            context=context+'\n\n--- WEB RESULTS (use only if the papers above lack the answer) ---\n\n'+web_context
    #
    response=client.chat.completions.create(
        model=model,
        max_tokens=500,
        messages=[
            {'role':'system','content':SYSTEM_PROMPT},
            {'role':'user','content':f'Context:\n{context}\n\nQuestion: {query}'}
        ]
    )
    return response.choices[0].message.content,retrieved,web_results,web_search_used


In [19]:
answer,sources,web_results,web_search_used=ask('What is Machine Learning?')
print(answer)
print('\nSources Used:')
for s in sources:
    print(f'[{s['score']:.3f}] {s['doc_title']}')
if web_search_used:
    print(f'\nWeb search triggered (best retrieval score below {WEB_SEARCH_THRESHOLD}).')
    print('Web Results:')
    for r in web_results:
        print(f"- {r['title']}\n  {r['url']}")
else:
    print('\nWeb search not needed: the corpus covered the question.')


[[-1.99545342e-02  9.87804495e-03  1.02495849e-02  2.95536984e-02
   2.71864664e-02 -1.92965437e-02 -2.41295770e-02 -3.77357379e-02
  -4.10542190e-02 -1.47499680e-03 -7.60767534e-02  3.68720740e-02
   5.37238345e-02 -7.05352351e-02 -8.60934481e-02  2.10068263e-02
  -4.22633477e-02  2.84489095e-02 -3.83753516e-02 -6.59191757e-02
   1.39496168e-02  1.22753698e-02 -6.82197288e-02  2.86251027e-02
   1.47611359e-02  3.22314762e-02  8.69424921e-03  1.30620906e-02
  -1.57267004e-02  1.88965611e-02  2.00238712e-02 -3.92971039e-02
   1.56471618e-02  3.33980396e-02 -7.33910203e-02  5.93578145e-02
  -1.23858228e-02  2.45846454e-02  2.01070774e-02 -2.02545002e-02
  -5.02477251e-02 -9.04411823e-02 -1.53743783e-02 -3.61244846e-03
   1.29996881e-01  1.16227120e-01 -5.74978292e-02 -1.04822204e-01
  -1.51266111e-02  8.75843968e-03 -1.15636416e-01 -2.95548700e-02
  -2.35043690e-02  1.07810851e-02 -4.12903242e-02  3.36975977e-02
   5.69917671e-02 -1.55181519e-03  3.50343110e-03 -1.57816224e-02
   2.62939

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01km0d59anebzb10b0xvef35yx` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198430, Requested 1735. Please try again in 1m11.28s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}